# 04 — Modelado: Aprendizaje Supervisado y No Supervisado

Este notebook implementa ambos enfoques de aprendizaje automatico:
- **Supervisado**: Random Forest Regressor para predecir calificaciones
- **No supervisado**: K-Means para agrupar peliculas por similitud

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.models.train_model import train_model
from src.models.evaluate_model import evaluate_regression
from src.models.predict import predict
from src.models.cluster import (
    train_kmeans,
    get_cluster_info,
    get_cluster_movies,
    save_model,
    load_model,
    prepare_features,
)

## 1. Cargar datos procesados

In [ ]:
df = pd.read_csv('../data/processed/peliculas_clean.csv')
print(f'Dataset: {df.shape[0]} peliculas, {df.shape[1]} columnas')
df.head()

## 2. Aprendizaje Supervisado — Random Forest

In [ ]:
model_sup, X_test, y_test = train_model(df, target_col='averageRating')
y_pred = predict(model_sup, X_test)
metrics = evaluate_regression(y_test, y_pred)
print('Metricas de regresion:')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')

## 3. Aprendizaje No Supervisado — K-Means Clustering

In [ ]:
# Entrenar K-Means con 5 clusters
model_km, scaler, df_clustered = train_kmeans(df, n_clusters=5)
print(f'Clusters formados: {model_km.n_clusters}')
print(f'Inercia (inertia): {model_km.inertia_:.2f}')

In [ ]:
# Resumen de cada cluster
cluster_summary = get_cluster_info(df_clustered)
cluster_summary

In [ ]:
# Distribucion de peliculas por cluster
df_clustered['cluster'].value_counts().sort_index().plot(
    kind='bar', figsize=(10, 5), color=['#22d3ee', '#8b5cf6', '#f472b6', '#34d399', '#facc15']
)
plt.title('Distribucion de peliculas por cluster')
plt.xlabel('Cluster')
plt.ylabel('Cantidad de peliculas')
plt.tight_layout()
plt.show()

In [ ]:
# Top 5 peliculas del cluster 0 (mas votadas)
get_cluster_movies(df_clustered, cluster_label=0, top_n=5)

## 4. Guardar modelo K-Means

In [ ]:
save_model(model_km, scaler, '../models/kmeans_model.joblib')
print('Modelo guardado en models/kmeans_model.joblib')

## 5. Verificar modelo cargado

In [ ]:
loaded_model, loaded_scaler = load_model('../models/kmeans_model.joblib')
print(f'Clusters del modelo cargado: {loaded_model.n_clusters}')
print(f'Centroides:\n{loaded_model.cluster_centers_}')